# **Task 2 — DeiT**: does seeing more pixels help?

This week's seminar was about [DeiT](https://proceedings.mlr.press/v139/touvron21a.html): a ViT trained on ImageNet alone, with distillation, and fine-tuned at a higher resolution at the end. The paper reports that going from 224×224 to 384×384 images makes the model more accurate (Table 10). Today you check whether that holds for **our** model and **our** data.

This notebook loads the data and the model and gives you a **fixed evaluation**, so that every group measures its result the same way. Your own work starts after the baseline.

<details>
<summary>Reach out here if you have any problems</summary>
Taras Kutsyk — taras.kutsyk@doctoral.uj.edu.pl
</details>

# The task

**Question.** Does seeing more pixels help? Make the model run at 384×384, fine-tune it within the budget, and **answer the question for this model and this data, with evidence**. Then explain *why* you got the result you got, and what they teach you.

*Hint:* be wary of what you compare with, and of how your claim relates to the paper's.

**Fixed:**
- the starting model and the dataset;
- the `evaluate()` function below;
- the fine-tuning budget: **at most 100,000 training images per fine-tuned model** (about 11 passes over the training set);
- write the resizing of the position embeddings yourselves: no ready-made resizing (e.g. timm's `img_size=` argument or `resample_abs_pos_embed`).

Everything else is your choice.

**Bring to the lab:** your code + a short deck with your answer, the evidence for it, and your explanation. If your group defends this week, **every member should be able to explain any piece of the code**.

# Setup

**Step 0.** Runtime → Change runtime type → **T4 GPU**. Everything we need (PyTorch, torchvision, timm) is already installed on Colab.

**Step 1. Data.** [Imagewoof](https://github.com/fastai/imagenette): 10 dog breeds from ImageNet, 9,025 training and 3,929 validation photos, **at full size** (about 1.3 GB, under a minute to download).

In [6]:
import os, tarfile, time, urllib.request
import torch, timm
from torchvision import datasets

device = "cuda" if torch.cuda.is_available() else "cpu"
assert device == "cuda", "Switch to a GPU runtime: Runtime -> Change runtime type -> T4 GPU"

URL = "https://s3.amazonaws.com/fast-ai-imageclas/imagewoof2.tgz"
# we use the full-size image sets on purpose, not the smaller "-320" versions
# can you tell why? (in the deck)
if not os.path.exists("imagewoof2/val"):
    urllib.request.urlretrieve(URL, "imagewoof2.tgz")
    with tarfile.open("imagewoof2.tgz") as t:
        t.extractall(filter="data")
for split in ["train", "val"]:
    print(split, sum(len(f) for _, _, f in os.walk(f"imagewoof2/{split}")), "photos")

AssertionError: Switch to a GPU runtime: Runtime -> Change runtime type -> T4 GPU

**Step 2. Model.** A distilled DeiT-S from the paper (12 blocks, 384 dims, 22M parameters, the DeiT-S⚗ of Table 5), trained on ImageNet at 224×224, together with the exact preprocessing it was trained with.

The model predicts one of 1000 ImageNet classes, so we map each Imagewoof folder (e.g. `n02086240` = Shih-Tzu) to its **ImageNet index**: the position of that breed among the model's 1000 outputs. You will need `to_imagenet` for your fine-tuning too.

<details>
<summary>Running the cell below can ask for your HF_TOKEN secret</summary>
Just press "Grant Access", it's automatically generated by Colab
</details>

In [ ]:
model = timm.create_model("deit_small_distilled_patch16_224", pretrained=True).to(device).eval()
cfg = timm.data.resolve_data_config({}, model=model)  # the preprocessing the model was trained with

# ImageFolder labels the 10 folders 0..9, but the model outputs 1000 ImageNet classes:
# to_imagenet[label] translates a folder label into the ImageNet class index
wnids = list(timm.data.ImageNetInfo().label_names())
breeds = datasets.ImageFolder("imagewoof2/val").classes
to_imagenet = torch.tensor([wnids.index(w) for w in breeds], device=device)
print(cfg)
print("folder -> ImageNet index:", dict(zip(breeds, to_imagenet.tolist())))

**Step 3. The fixed evaluation** (DON'T modify it). `evaluate(model, resolution)` resizes the validation photos for the given resolution exactly as the model's own preprocessing does, and returns the **top-1 accuracy over the 10 breeds**: the model's prediction is the highest-scoring of its 10 dog outputs.

In [ ]:
@torch.no_grad()
def evaluate(model, resolution):
    """The fixed evaluation for task 2. Do not change it; report exactly what it prints.

    model      - any model that takes (batch, 3, resolution, resolution) images and returns 1000 logits
    resolution - the input size: 224 for the original model, e.g. 384 for yours
    returns: accuracy - top-1 over the 10 breeds, on the 3,929 Imagewoof validation photos
    """
    model.eval()
    tf = timm.data.create_transform(**dict(cfg, input_size=(3, resolution, resolution)))
    loader = torch.utils.data.DataLoader(datasets.ImageFolder("imagewoof2/val", transform=tf),
                                         batch_size=64, num_workers=2)
    correct = n = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(x)
        correct += (logits[:, to_imagenet].argmax(-1) == y).sum().item()  # best of the 10 breeds
        n += len(y)
    return correct / n

**Step 4. Baseline.** Takes about half a minute. You should get **top-1 ≈ 97.20%** at 224. If not, something in the setup went wrong.

In [ ]:
print(f"baseline DeiT-S distilled @224: top-1 {evaluate(model, 224):.2%}")

# Before you start

Think about **what exactly breaks in the model when you pass it higher-resolution images?** What's the simplest, most effective way to fix that?

Training photos are in `imagewoof2/train` (same folder structure as `val`).

# Your solution

Make the model run at 384, fine-tune within the budget, and collect the evidence for your answer. Good luck!

In [ ]:
# your_model = ...
# print(f"ours @384: top-1 {evaluate(your_model, 384):.2%}")

In [ ]:
print("pos_embed:", model.pos_embed.shape)
print("ile tokenow specjalnych:", model.num_prefix_tokens)
print("patch_embed:", model.patch_embed)
print("img_size:", model.patch_embed.img_size, "| grid_size:", model.patch_embed.grid_size)
print("pierwsze 5 liczb pozycji [CLS]:", model.pos_embed[0, 0, :5])
print("pierwsze 5 liczb pozycji [DIST]:", model.pos_embed[0, 1, :5])

In [ ]:
x = torch.randn(1, 3, 384, 384, device=device)
model(x)

In [ ]:
import copy
import copy, math
import torch.nn as nn
import torch.nn.functional as F

def resize_pos_embed(pos_embed, num_prefix_tokens, new_grid):
    # 1. odetnij tokeny specjalne, siatkę zostaw osobno
    prefix = pos_embed[:, :num_prefix_tokens]          # (1, 2, 384)
    grid = pos_embed[:, num_prefix_tokens:]            # (1, 196, 384)
    old_grid = int(math.sqrt(grid.shape[1]))           # 14
    dim = grid.shape[-1]                               # 384

    # 2. lista -> siatka 2D, kanały na przód
    grid = grid.reshape(1, old_grid, old_grid, dim)    # (1, 14, 14, 384)
    grid = grid.permute(0, 3, 1, 2)                    # (1, 384, 14, 14)

    # 3. interpolacja każdego z 384 "obrazków" 14x14 -> 24x24
    grid = F.interpolate(grid, size=(new_grid, new_grid),
                         mode="bicubic", align_corners=False)   # (1, 384, 24, 24)

    # 4. z powrotem do listy
    grid = grid.permute(0, 2, 3, 1)                    # (1, 24, 24, 384)
    grid = grid.reshape(1, new_grid * new_grid, dim)   # (1, 576, 384)

    # 5. doklej tokeny specjalne z powrotem na początek
    return torch.cat([prefix, grid], dim=1)            # (1, 578, 384)

In [ ]:
pe = model.pos_embed.data
g = pe[:, 2:]
print("lista:            ", tuple(g.shape))
g = g.reshape(1, 14, 14, 384); print("po reshape:       ", tuple(g.shape))
g = g.permute(0, 3, 1, 2);     print("po permute:       ", tuple(g.shape))
print("cała funkcja daje:", tuple(resize_pos_embed(pe, 2, 24).shape))

In [7]:
t = torch.arange(8).reshape(2, 2, 2)      # (kanały, H, W); kanał 0 = 0..3, kanał 1 = 4..7
print(t)
print("zle:   ", t.reshape(4, 2).tolist())
print("dobrze:", t.permute(1, 2, 0).reshape(4, 2).tolist())

tensor([[[0, 1],
         [2, 3]],

        [[4, 5],
         [6, 7]]])
zle:    [[0, 1], [2, 3], [4, 5], [6, 7]]
dobrze: [[0, 4], [1, 5], [2, 6], [3, 7]]


In [8]:
def at_resolution(src, resolution):
    m = copy.deepcopy(src)                       # 1. pracujemy na kopii
    g = resolution // 16                         # 384 // 16 = 24
    with torch.no_grad():                        # 2. to "chirurgia", nie trening
        new_pe = resize_pos_embed(m.pos_embed.data, m.num_prefix_tokens, g)
    m.pos_embed = nn.Parameter(new_pe)           # 3. nowa tabela jako trenowalny parametr
    m.patch_embed.img_size = (resolution, resolution)   # 4. "bramkarz" wpuści 384
    m.patch_embed.grid_size = (g, g)
    m.patch_embed.num_patches = g * g
    return m



pe = model.pos_embed.data.clone()
assert torch.allclose(resize_pos_embed(pe, 2, 14), pe), "14 -> 14 musi nic nie zmieniac"

m384 = at_resolution(model, 384).to(device).eval()
print("pos_embed:", tuple(model.pos_embed.shape), "->", tuple(m384.pos_embed.shape))
assert torch.equal(m384.pos_embed[:, :2], pe[:, :2]), "[CLS]/[DIST] musza zostac bez zmian"
print("wyjscie dla 384:", tuple(m384(torch.randn(2, 3, 384, 384, device=device)).shape))
assert torch.equal(model.pos_embed, pe), "oryginal musi zostac nietkniety"

print(f"R1 (384, bez dotrenowania): top-1 {evaluate(m384, 384):.2%}")

NameError: name 'model' is not defined

In [ ]:
import time, random, math
import numpy as np
from torchvision import transforms
from torchvision.transforms import InterpolationMode

SMOKE_TEST = False          # najpierw szybka próba, potem zmienimy na False
BUDGET     = 100_000       # max zdjęć treningowych na jeden model (z PDF-a)
BATCH_SIZE = 32
LR         = 2e-5
if SMOKE_TEST:
    BUDGET = 640           # 20 kroków: tylko sprawdzamy, czy wszystko działa

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def train_transform(resolution):
    return transforms.Compose([
        transforms.RandomResizedCrop(resolution, scale=(0.25, 1.0),
                                     interpolation=InterpolationMode.BICUBIC),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(cfg["mean"], cfg["std"]),
    ])

In [ ]:
def lr_at(step, total):
    warm = max(1, int(0.05 * total))                  # pierwsze 5% kroków: rozgrzewka
    if step < warm:
        return LR * (step + 1) / warm
    p = (step - warm) / max(1, total - warm)
    return LR * 0.5 * (1 + math.cos(math.pi * p))     # potem łagodne wygaszanie do 0

def finetune(resolution, seed):
    set_seed(seed)
    m = at_resolution(model, resolution).to(device)   # świeża kopia (przy 224 nic się nie zmienia)
    ds = datasets.ImageFolder("imagewoof2/train", transform=train_transform(resolution))
    loader = torch.utils.data.DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True,
                                         num_workers=2, generator=torch.Generator().manual_seed(seed))
    total = BUDGET // BATCH_SIZE                      # 3125 kroków (w smoke teście 20)
    log_every = max(1, total // 25)
    opt = torch.optim.AdamW(m.parameters(), lr=LR, weight_decay=0.05)
    scaler = torch.amp.GradScaler()
    step, seen, t0 = 0, 0, time.time()
    m.train()
    while step < total:                               # kolejne przejścia przez zbiór
        for x, y in loader:                           # paczka 32 zdjęć
            if step == total:
                break
            x, y = x.to(device), y.to(device)
            for g in opt.param_groups:
                g["lr"] = lr_at(step, total)
            # --- jeden krok uczenia ---
            with torch.autocast("cuda", dtype=torch.float16):
                logits = m(x)[:, to_imagenet]         # (32, 10): tylko 10 ras
            loss = F.cross_entropy(logits.float(), y, label_smoothing=0.1)
            opt.zero_grad()
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            # ---------------------------
            step += 1
            seen += len(y)
            if step % log_every == 0 or step == total:
                acc = (logits.argmax(-1) == y).float().mean().item()
                print(f"krok {step:>4}/{total} | zdjec {seen:>7,}/{BUDGET:,} | loss {loss.item():.3f} "
                      f"| trafnosc w paczce {acc:>4.0%} | {time.time() - t0:>5.0f} s")
    assert seen <= BUDGET, "przekroczony budzet!"
    m.eval()
    return m

In [ ]:
# m_test = finetune(384, seed=0)
# print(f"smoke test @384: top-1 {evaluate(m_test, 384):.2%}")
# del m_test

In [ ]:
import json
results = {}

def record(name, m, res):
    acc = evaluate(m, res)                              # oficjalna liczba
    results[name] = acc
    print(f">>> {name}: top-1 {acc:.2%}")
    with open("results.json", "w") as f:                # zapis po każdym wyniku
        json.dump(results, f, indent=2)

record("R0 224 bez treningu", model, 224)
record("R1 384 bez treningu", m384, 384)

SEEDS = [0, 1]
for seed in SEEDS:
    for res in [224, 384]:
        name = f"{'R2' if res == 224 else 'R3'} {res} dotrenowany s{seed}"
        print(f"\n===== {name} =====")
        m = finetune(res, seed)
        record(name, m, res)
        del m
        torch.cuda.empty_cache()                        # zwalniamy pamięć GPU

In [ ]:
r0 = results["R0 224 bez treningu"]
r1 = results["R1 384 bez treningu"]
r2 = np.array([results[f"R2 224 dotrenowany s{s}"] for s in SEEDS])
r3 = np.array([results[f"R3 384 dotrenowany s{s}"] for s in SEEDS])

print(f"{'model':<26}{'top-1':>8}")
for k, v in results.items():
    print(f"{k:<26}{v:>8.2%}")
print()
print(f"R3 - R2 (wiecej pikseli):          {100 * (r3.mean() - r2.mean()):+.2f} pkt   "
      f"per seed: {[f'{100 * d:+.2f}' for d in r3 - r2]}")
print(f"R2 - R0 (samo dotrenowanie):       {100 * (r2.mean() - r0):+.2f} pkt")
print(f"R3 - R0 (naiwne porownanie):       {100 * (r3.mean() - r0):+.2f} pkt")
print(f"R1 - R0 (samo przeskalowanie):     {100 * (r1 - r0):+.2f} pkt")
print(f"1 zdjecie = {100 / 3929:.3f} pkt")

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({"font.size": 13, "axes.spines.top": False, "axes.spines.right": False})
BLUE, ORANGE, GRAY, INK = "#3B6EA8", "#C8692A", "#6B7280", "#14213D"
N = 3929
def se(p):                       # blad standardowy dokladnosci p na N zdjeciach
    return math.sqrt(p * (1 - p) / N)

In [ ]:
keys = ["R0 224 bez treningu", "R1 384 bez treningu", "R2 224 dotrenowany s0",
        "R2 224 dotrenowany s1", "R3 384 dotrenowany s0", "R3 384 dotrenowany s1"]
labels = ["R0\n224, original", "R1\n384, no FT", "R2 s0\n224, FT", "R2 s1\n224, FT",
          "R3 s0\n384, FT", "R3 s1\n384, FT"]
cols = [GRAY, GRAY, BLUE, BLUE, ORANGE, ORANGE]

fig, ax = plt.subplots(figsize=(10, 5))
for i, (k, c) in enumerate(zip(keys, cols)):
    p = 100 * results[k]
    ax.errorbar(i, p, yerr=100 * se(results[k]), fmt="o", color=c, ms=10, capsize=6, lw=2)
    ax.annotate(f"{p:.2f}%", (i, p), xytext=(12, 0), textcoords="offset points", va="center", color=INK)
ax.axhline(100 * results[keys[0]], color=GRAY, ls="--", lw=1)
ax.text(5.45, 100 * results[keys[0]], "R0", color=GRAY, va="center")
ax.set_xticks(range(len(keys))); ax.set_xticklabels(labels)
ax.set_ylabel("top-1 (%), evaluate()")
ax.set_title("Accuracy on 3,929 validation photos (± 1 standard error)")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.savefig("chart_results.png", dpi=200); plt.show()

In [ ]:
diffs = 100 * (r3 - r2)
se_d = 100 * math.sqrt(2) * se(r2.mean())        # ostrozny SE roznicy dwoch modeli
fig, ax = plt.subplots(figsize=(10, 3.4))
ax.axvspan(-2 * se_d, 2 * se_d, color=GRAY, alpha=0.12, label="± 2 SE: could be chance")
ax.axvspan(-se_d, se_d, color=GRAY, alpha=0.20, label="± 1 SE")
ax.axvline(0, color=INK, lw=1)
for i, d in enumerate(diffs):
    ax.plot(d, i + 1, "o", color=ORANGE, ms=10)
    ax.annotate(f"seed {SEEDS[i]}: {d:+.2f}", (d, i + 1), xytext=(10, 0), textcoords="offset points", va="center")
ax.plot(diffs.mean(), 0, "D", color=INK, ms=11)
ax.annotate(f"mean: {diffs.mean():+.2f} pts", (diffs.mean(), 0), xytext=(10, 0), textcoords="offset points", va="center", weight="bold")
ax.set_yticks([]); ax.set_ylim(-0.7, len(diffs) + 0.7)
ax.set_xlabel("R3 − R2 (percentage points): effect of 384 with identical fine-tuning")
ax.legend(loc="lower right", frameon=False)
plt.tight_layout(); plt.savefig("chart_answer.png", dpi=200); plt.show()

In [ ]:
def sim_map(grid, i, j):
    g = grid.shape[0]
    flat = grid.reshape(g * g, -1)
    return F.cosine_similarity(flat, grid[i, j][None], dim=1).reshape(g, g).cpu().numpy()

old = model.pos_embed.detach()[0, 2:].reshape(14, 14, -1).float()
new = m384.pos_embed.detach()[0, 2:].reshape(24, 24, -1).float()
spots = [("top-left", (1, 1), (2, 2)), ("centre", (7, 7), (12, 12)), ("bottom-right", (12, 12), (21, 21))]
fig, axs = plt.subplots(2, 3, figsize=(11, 7.5))
for c, (name, (oi, oj), (ni, nj)) in enumerate(spots):
    for r, (grid, (i, j), title) in enumerate([(old, (oi, oj), "learned 14×14"), (new, (ni, nj), "ours, 24×24")]):
        axs[r, c].imshow(sim_map(grid, i, j), cmap="RdBu_r", vmin=-1, vmax=1)
        axs[r, c].plot(j, i, "x", color=INK, ms=10, mew=2)
        axs[r, c].set_title(f"{title}: {name}", fontsize=12)
        axs[r, c].axis("off")
fig.suptitle("Cosine similarity of one position (×) to all others", fontsize=14)
plt.tight_layout(); plt.savefig("chart_pos_embed.png", dpi=200); plt.show()

In [ ]:
items = ["Training time\n(21.1 vs 9.6 min)", "Tokens\n(578 vs 198)", "Attention pairs / layer\n(334,084 vs 39,204)"]
ratios = [1269 / 578, 578 / 198, 578 ** 2 / 198 ** 2]
fig, ax = plt.subplots(figsize=(10, 3.6))
ax.barh(items, ratios, color=ORANGE, height=0.5)
for i, v in enumerate(ratios):
    ax.text(v + 0.1, i, f"× {v:.1f}", va="center", weight="bold", color=INK)
ax.axvline(1, color=GRAY, ls="--", lw=1)
ax.set_xlabel("384 relative to 224 (× times)")
ax.invert_yaxis()
plt.tight_layout(); plt.savefig("chart_cost.png", dpi=200); plt.show()